In [2]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# -----------------------------
# 1. CREATE SYNTHETIC DATA (MATCHING YOUR UI)
# -----------------------------

np.random.seed(42)
n_samples = 2000

df = pd.DataFrame({
    "production_volume": np.random.randint(500, 5000, n_samples),
    "production_cost": np.random.randint(20000, 100000, n_samples),
    "defect_rate": np.random.uniform(0, 10, n_samples),
    "quality_score": np.random.uniform(50, 100, n_samples),
    "maintenance_hours": np.random.uniform(0, 50, n_samples),
    "stockout_rate": np.random.uniform(0, 5, n_samples),
    "energy_efficiency": np.random.uniform(0.5, 1.0, n_samples)
})

# -----------------------------
# 2. CREATE TARGET (SMART LOGIC)
# -----------------------------

df["defective"] = (
    (df["defect_rate"] > 5) |
    (df["quality_score"] < 70) |
    (df["maintenance_hours"] > 30) |
    (df["energy_efficiency"] < 0.65)
).astype(int)

print("Original Distribution:\n", df["defective"].value_counts())

# -----------------------------
# 3. SPLIT
# -----------------------------

X = df.drop("defective", axis=1)
y = df["defective"]

# -----------------------------
# 4. BALANCE DATA (SMOTE)
# -----------------------------

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("\nBalanced Distribution:\n", pd.Series(y_resampled).value_counts())

# -----------------------------
# 5. TRAIN TEST SPLIT
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# -----------------------------
# 6. SCALING
# -----------------------------

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# 7. MODEL TRAINING
# -----------------------------

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------
# 8. EVALUATION
# -----------------------------

y_pred = model.predict(X_test)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# -----------------------------
# 9. TEST SAMPLE (LIKE FRONTEND INPUT)
# -----------------------------

sample_input = [[
    1500,   # production_volume
    45000,  # production_cost
    3.2,    # defect_rate
    87,     # quality_score
    12,     # maintenance_hours
    1.5,    # stockout_rate
    0.78    # energy_efficiency
]]

sample_scaled = scaler.transform(sample_input)

prob = model.predict_proba(sample_scaled)[0]
pred = model.predict(sample_scaled)[0]

print("\nSample Prediction:")
print("Result:", "Defective" if pred == 1 else "Non-Defective")
print(f"Confidence: {max(prob)*100:.2f}%")
print("Probabilities:", prob)

# -----------------------------
# 10. SAVE MODEL
# -----------------------------

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("\nModel saved successfully!")

Original Distribution:
 defective
1    1750
0     250
Name: count, dtype: int64

Balanced Distribution:
 defective
1    1750
0    1750
Name: count, dtype: int64

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       330
           1       0.99      1.00      1.00       370

    accuracy                           1.00       700
   macro avg       1.00      1.00      1.00       700
weighted avg       1.00      1.00      1.00       700


Sample Prediction:
Result: Non-Defective
Confidence: 100.00%
Probabilities: [1. 0.]

Model saved successfully!


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
